[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/EelcoHoogendoorn/numga/blob/main/examples/electromagnetism/superconductor/superconductor.ipynb)

# The Meissner Effect

This example computes magnetic field expulsion in a superconducting rod cooled through its critical temperature in an applied magnetic field, alongside magnetic flux quantization in a superconducting ring. Below the critical temperature, conduction electrons are modelled as two interpenetrating fluids, a resistive normal fluid and a dissipationless superfluid whose fraction grows as temperature falls ([Stockholm superconductivity lecture notes](https://ekmf.fysik.su.se/Supraledning/WWW/Lectures/Lecture3.pdf)). Whereas a hypothetical metal with zero resistance merely freezes in whatever magnetic flux threaded it during cooling, a superconductor develops surface screening currents that expel interior flux down to a thin boundary layer of thickness equal to the London penetration depth, an expulsion called the Meissner effect. Around a multiply connected ring, single-valuedness of the superfluid condensate phase quantizes the combined magnetic flux and supercurrent circulation, so cooling a ring in an external field traps an integer number of flux quanta once the external field is removed.

In spacetime, the electromagnetic potential is a 1-form whose exterior derivative `gradient ^ potential` gives the Faraday bivector, while the vacuum duality map sends that field bivector to its excitation antibivector for Ampère's law. Because the canonical momentum of the superfluid is the spacetime gradient of one condensate phase, its exterior derivative vanishes, `gradient ^ (momentum + charge * potential) == 0`; the space-time components of this single closure accelerate supercurrent under an electric field, while the spatial components lock supercurrent directly to the potential and drive Meissner screening. On each triangle of the planar cross-section, the term ***simplex gradient*** here means the spatial plane `(rod_direction & time & sides) / double_areas` that pairs to unity from the facing side to its corner, and rest-frame superfluid inertia `t | (Current & time).dual()` extracts the spatial momentum 1-form of a spacetime 4-current without singling out spatial coordinate axes.

In Gibbs–Heaviside vector notation, that single phase-closure condition splits into London's first equation $\partial_t(\Lambda\mathbf{J}_s) = \mathbf{E} = -\nabla\phi - \partial_t\mathbf{A}$ for electric acceleration and London's second equation $\nabla\times(\Lambda\mathbf{J}_s) = -\mathbf{B} = -\nabla\times\mathbf{A}$ for magnetic screening, which combines with the quasistatic Ampère curl $\nabla\times\mathbf{H} = \mathbf{J}_n + \mathbf{J}_s$ and vector identity $\nabla\times(\nabla\times\mathbf{H}) = \nabla(\nabla\cdot\mathbf{H}) - \nabla^2\mathbf{H}$ into $\lambda_L^2\nabla\times(\nabla\times\mathbf{H}) + \mathbf{H} = \mathbf{0}$, while fluxoid quantization around a ring couples a surface integral to a boundary line integral via $\iint_S \mathbf{B}\cdot\mathbf{n}\,\mathrm{d}A + \Lambda\oint_{\partial S}\mathbf{J}_s\cdot\mathrm{d}\mathbf{l} = n\Phi_0$.

In [ ]:
# In Colab, numga and its examples, from GitHub.
import sys

if "google.colab" in sys.modules:
    %pip install -q "numga[examples] @ https://github.com/EelcoHoogendoorn/numga/archive/HEAD.tar.gz"

In [ ]:
%matplotlib inline
from collections.abc import Iterator

import numpy as np

from examples.electromagnetism.superconductor import render, scenarios
from examples.util.animation import show_states
from numga import Algebra, NumpyContext, stack
from numga.algebras import STA as ga
from numga.sparse import spdiag
from numga.topology import Topology

mv = NumpyContext(ga).multivector
Scalar = ga.gatype.scalar()                                                   # Scalar+
Plane = ga.gatype.vector().derive.within(Algebra("x+y+z+"))                   # Vector+
Point = Plane.derive.dual                                                     # Antivector-
Excitation = ga.gatype.antibivector()                                         # Antibivector-
Field = Excitation.derive.dual                                                # Bivector+
Current = ga.gatype.antivector()                                              # Antivector-

# Symmetry directions along the rod and rest-frame clock, vacuum constitutive duality, and rest-frame superfluid inertia.
rod_direction = mv.z.dual()                                                   # [] Point-
time = -mv.t.dual()                                                           # [] Current-
vacuum = Field.dual()                                                         # [] Excitation- <- Field+
axial_excitation = vacuum(mv.xy)                                              # [] Excitation-
inertia = mv.t | (Current & time).dual()                                      # [] Plane+ <- Current-

## Cross-section of the rod

The rod cross-section has unit radius inside a concentric outer disk of twice that radius, triangulated in rings so one ring of edges lies along the rod boundary. On the outer rim an axial potential proportional to vertical height imposes a uniform horizontal magnetic field across the domain. Because fields do not vary along the rod or in time for static geometry, sweeping each triangle's opposite side along `rod_direction` and `time` and dividing by twice the signed area yields each corner's simplex gradient. Lengths are measured in rod radii, magnetic fields in units of the initial applied field, and times in normal-metal magnetic diffusion times.

In [ ]:
def triangle_areas(topology: Topology, vertices: Point) -> Scalar:
    """Signed side joins swept along the rod and through time give each triangle's area (`Scalar+[F]`)."""
    sides = topology.boundary[2, 1] * (topology.d[0] * vertices)              # [3] Point-[F]
    double_areas = ((sides[0] & sides[1]) & rod_direction) & time             # Scalar-[F]
    return double_areas.norm() / 2


def simplex_gradients(topology: Topology, vertices: Point) -> Plane:
    """Each corner's simplex gradient (`[3] Plane+[F]`) pairs to one from the facing side to that corner."""
    sides = topology.boundary[2, 1] * (topology.d[0] * vertices)              # [3] Point-[F]
    double_areas = ((sides[0] & sides[1]) & rod_direction) & time             # Scalar-[F]
    return (rod_direction & time & sides) / double_areas


topology, coordinates, in_rod = scenarios.rod_scene(scenarios.n_divisions)
vertices = mv(Plane, coordinates).dual().field()                              # Point-[V]
areas = triangle_areas(topology, vertices)                                    # Scalar+[F]
gradients = simplex_gradients(topology, vertices)                             # [3] Plane+[F]
render.rod_figure(topology, vertices, in_rod);

The triangulated domain surrounds the pale grey rod cross-section, with a ring of triangle edges aligned along the circular boundary between metal and surrounding vacuum.

## Cooling in an applied field

Starting in the normal state where the applied field penetrates uniformly, the rod cools below its critical temperature while the superfluid electron fraction grows, holds cold in the applied field, and then experiences a linear drop to zero external field. At each implicit time step, Ampère stiffness of the corner magnetic fields balances normal Ohmic eddy currents driven by time-varying potential together with the London supercurrent. Because condensate phase does not vary along the rod, the superconductor's canonical momentum along the rod vanishes and its supercurrent responds to total potential, whereas the comparison perfect conductor responds only to departures from the potential present when its lossless fluid formed. Running the cooling schedule and rendering the side-by-side animation takes about fifteen seconds.

In [ ]:
penetration_depth = scenarios.penetration_depth
superfluid_shares, applied_fields = scenarios.cooling_schedule()              # [n_steps], [n_steps]
dt, n_substeps = scenarios.dt, scenarios.n_substeps

In [ ]:
def cooling_states(
    topology: Topology,
    areas: Scalar,
    gradients: Plane,
    rod_mask: Scalar,
    heights: Scalar,
    reference_potential: Scalar,
    penetration_depth: float,
    superfluid_shares: np.ndarray,
    applied_fields: np.ndarray,
    dt: float,
    n_substeps: int,
) -> Iterator[tuple[Scalar, Scalar]]:
    """Yields nodal potential and supercurrent (`[...] Scalar+[V]`) every `n_substeps` cooling steps of `dt`."""
    corners = topology.incidence[2, 0]                                        # [3] Scalar+[F <- V]
    dual_areas = (corners.adjugate() * (areas / 3)).sum(axis=0)               # Scalar+[V]
    free = topology.reading(topology.interior[0], 0)                          # Scalar+[V_free <- V]
    # Ampère stiffness pairs each corner's axial-potential field bivector with the vacuum excitation of another.
    corner_fields = gradients ^ mv.z                                          # [3] Field+[F]
    local_stiffness = (corner_fields[:, None] & vacuum(corner_fields[None, :])) * areas   # [3, 3] Scalar+[F]
    stiffness = (corners[:, None].adjugate() * spdiag(local_stiffness) * corners[None, :]).sum(axis=0).sum(axis=0)   # Scalar+[V <- V]
    conduction = spdiag(dual_areas * rod_mask / dt)                           # Scalar+[V <- V]
    potential = applied_fields[0] * heights + 0 * reference_potential         # [...] Scalar+[V]
    for step_index, (share, field_strength) in enumerate(zip(superfluid_shares, applied_fields)):
        london_weight = spdiag(dual_areas * rod_mask * share / penetration_depth ** 2)   # Scalar+[V <- V]
        boundary_potential = field_strength * heights                         # Scalar+[V]
        step_operator = conduction + stiffness + london_weight                # Scalar+[V <- V]
        interior_system = (free * step_operator * free.adjugate()) * Scalar   # (Scalar+ <- Scalar+)[V_free <- V_free]
        rhs = conduction * potential + london_weight * reference_potential - step_operator * boundary_potential
        potential = boundary_potential + free.adjugate() * interior_system.solve(free * rhs)   # [...] Scalar+[V]
        if step_index % n_substeps == 0:
            supercurrent = -(rod_mask * share / penetration_depth ** 2) * (potential - reference_potential)
            yield potential, supercurrent


rod_mask = mv.scalar(in_rod[:, None]).field()                                 # Scalar+[V]
heights = mv.y & vertices                                                     # Scalar+[V]
reference_potential = stack([0 * heights, heights])                           # [2] Scalar+[V]
recorded_states = show_states(
    cooling_states(
        topology, areas, gradients, rod_mask, heights, reference_potential,
        penetration_depth, superfluid_shares, applied_fields, dt, n_substeps,
    ),
    render.cooling_animation, topology, vertices, in_rod, penetration_depth,
    name="superconductor", duration_ms=scenarios.frame_duration_ms,
)

The superconductor on the left and the perfect conductor on the right are shaded by axial supercurrent, red flowing out of the cross-section and blue flowing into it, beneath black magnetic field contours. As the superconductor cools, surface screening currents build up and push the magnetic field lines outward around the rod at a pace set by normal-electron eddy damping, whereas the perfect conductor remains inactive while the field stays steady. Once the external field drops to zero, the superconductor relaxes back to a field-free state with no residual current, while the perfect conductor develops opposing surface currents that lock its initial interior flux into closed dipole loops.

## Penetration depth

At the end of the cold hold in the applied field, wedging the corner simplex gradients with the axial potential gives the magnetic field bivector across every triangle. Along the upper vertical radius where exterior field lines graze the rod horizontally, London screening predicts exponential decay of field strength with depth on the scale of the penetration depth.

In [ ]:
def magnetic_field(topology: Topology, gradients: Plane, potential: Scalar) -> Field:
    """Corner simplex gradients wedged with nodal axial potentials sum to the magnetic field (`[...] Field+[F]`)."""
    corners = topology.incidence[2, 0]                                        # [3] Scalar+[F <- V]
    return (gradients ^ ((corners * potential[..., None]) * mv.z)).sum(axis=-1)


def radial_profile(topology: Topology, vertices: Point, field: Field, radial_width: float) -> tuple[Scalar, Scalar]:
    """Triangles along the upper radius give surface depth and magnetic field strength (`[...] Scalar+`)."""
    corners = topology.incidence[2, 0]                                        # [3] Scalar+[F <- V]
    centroids = corners.mean(axis=0) * vertices                               # Point-[F]
    radii = centroids.antinorm()                                              # Scalar+[F]
    ray_mask = ((mv.x & centroids).abs() < radial_width) & ((mv.y & centroids) > 0) & (radii < 1)
    return (1 - radii).batch()[ray_mask], field.norm().batch()[ray_mask]


held_index = (scenarios.n_normal + scenarios.n_cooling + scenarios.n_held - 1) // n_substeps
held_potential = recorded_states[held_index][0][0]                            # Scalar+[V]
held_field = magnetic_field(topology, gradients, held_potential)              # Field+[F]
depths, strengths = radial_profile(topology, vertices, held_field, scenarios.radial_width)
render.penetration_figure(depths, strengths, penetration_depth);

Blue circles plot triangle field strengths in units of the applied field against depth in penetration depths, and the orange curve shows pure exponential decay anchored at the shallowest triangle. Crowding of expelled flux lines around the cylinder enhances surface field strength toward two thirds above the applied value, after which the field drops exponentially across nearly three decades before levelling into a faint eddy-current remnant near the axis.

## Flux quantization in a ring

Around a superconducting ring, condensate phase single-valuedness restricts the fluxoid, the magnetic flux threading the central hole plus the kinetic circulation of supercurrent around the hole boundary, to integer flux quanta. For an axial magnetic field perpendicular to the cross-section, field strength stands uniform across the central hole, screens into the ring according to London's equation, and shares its level contours with the circulating supercurrent lines. Wedging corner simplex gradients with the unit axial excitation gives each corner's Ampère supercurrent, and pairing those currents through rest-frame superfluid inertia `inertia(corner_currents) & corner_currents` assembles the kinetic screening operator whose boundary reaction at the inner edge supplies the circulation share of the fluxoid.

In [ ]:
ring_topology, ring_coordinates, in_hole = scenarios.ring_scene(scenarios.n_ring_divisions)
trapped_quanta = scenarios.trapped_quanta                                     # [3]
candidate_quanta = scenarios.candidate_quanta                                 # [n_candidates]
cooling_fields = scenarios.cooling_fields                                     # [n_cooling_fields]

In [ ]:
def ring_fields(
    topology: Topology,
    areas: Scalar,
    gradients: Plane,
    in_hole: np.ndarray,
    penetration_depth: float,
    quanta: np.ndarray,
    applied_fields: np.ndarray,
) -> Scalar:
    """London screening on a ring determines the axial field (`[...] Scalar+[V]`) from fluxoid quanta and rim fields."""
    V, E, F = topology.n_elements
    corners = topology.incidence[2, 0]                                        # [3] Scalar+[F <- V]
    hole_faces = in_hole[topology.elements[2]].all(axis=-1)                   # [F]
    ring_indicator = mv.scalar((~hole_faces)[:, None]).field()                # Scalar+[F]
    # Ampère supercurrents of unit corner axial fields pair with rest-frame inertia to assemble London screening.
    corner_currents = gradients ^ axial_excitation                            # [3] Current-[F]
    local_kinetic = (inertia(corner_currents[:, None]) & corner_currents[None, :]) * (ring_indicator * areas)   # [3, 3] Scalar+[F]
    kinetic_form = (corners[:, None].adjugate() * spdiag(local_kinetic) * corners[None, :]).sum(axis=0).sum(axis=0)   # Scalar+[V <- V]
    screening = penetration_depth ** 2 * kinetic_form + (corners.adjugate() * (ring_indicator * areas / 3)).sum(axis=0)   # Scalar+[V <- V]
    rim_mask = np.isin(np.arange(V), topology.exterior[0])                    # [V]
    free = topology.reading(np.flatnonzero(~(in_hole | rim_mask)), 0)         # Scalar+[V_free <- V]
    hole_unit = mv.scalar(in_hole[:, None]).field()                           # Scalar+[V]
    rim_unit = mv.scalar(rim_mask[:, None]).field()                           # Scalar+[V]
    boundary_units = stack([hole_unit, rim_unit])                             # [2] Scalar+[V]
    ring_system = (free * screening * free.adjugate()) * Scalar               # (Scalar+ <- Scalar+)[V_free <- V_free]
    basis_fields = boundary_units + free.adjugate() * ring_system.solve(-(free * (screening * boundary_units)))   # [2] Scalar+[V]
    # Each basis mode's fluxoid combines hole magnetic flux with inner-edge screening reaction.
    hole_area = (mv.scalar(hole_faces[:, None]).field() * areas).sites.sum()  # [] Scalar+
    basis_fluxoids = (hole_unit * (screening * basis_fields)).sites.sum() + hole_area * np.array([1, 0])   # [2] Scalar+
    hole_strength = (quanta - applied_fields * basis_fluxoids[1]) / basis_fluxoids[0]   # [...] Scalar+
    return hole_strength * basis_fields[0] + applied_fields * basis_fields[1]


ring_vertices = mv(Plane, ring_coordinates).dual().field()                    # Point-[V]
ring_areas = triangle_areas(ring_topology, ring_vertices)                     # Scalar+[F]
ring_gradients = simplex_gradients(ring_topology, ring_vertices)              # [3] Plane+[F]
trapped_fields = ring_fields(
    ring_topology, ring_areas, ring_gradients, in_hole,
    penetration_depth, trapped_quanta, 0 * trapped_quanta,
)                                                                             # [3] Scalar+[V]
render.trapped_figure(ring_topology, ring_vertices, in_hole, trapped_fields);

The three panels show one, two, and three trapped flux quanta in zero external field, shaded from white at zero axial field to deep purple at high field beneath black circulating supercurrent contours. Uniform magnetic flux fills the central hole and decays across a thin inner skin of the ring where persistent currents crowd, while the outer rim remains free of field.

When the ring cools through its transition inside an axial cooling field, it locks into whichever integer fluxoid minimizes the combined magnetic field departure energy and supercurrent kinetic energy, and retains that integer fluxoid after the external field is switched off.

In [ ]:
def equilibrium_quanta(
    topology: Topology,
    areas: Scalar,
    gradients: Plane,
    in_hole: np.ndarray,
    penetration_depth: float,
    candidate_quanta: np.ndarray,
    cooling_fields: np.ndarray,
) -> np.ndarray:
    """Energy minimization across candidate fluxoids picks the retained quantum number for each cooling field (`[...]`)."""
    corners = topology.incidence[2, 0]                                        # [3] Scalar+[F <- V]
    dual_areas = (corners.adjugate() * (areas / 3)).sum(axis=0)               # Scalar+[V]
    hole_faces = in_hole[topology.elements[2]].all(axis=-1)                   # [F]
    ring_areas = mv.scalar((~hole_faces)[:, None]).field() * areas            # Scalar+[F]
    fields = ring_fields(
        topology, areas, gradients, in_hole, penetration_depth,
        candidate_quanta[:, None], cooling_fields[None, :],
    )                                                                         # [...] Scalar+[V]
    slopes = (gradients * (corners * fields[..., None])).sum(axis=-1)         # [...] Plane+[F]
    currents = slopes ^ axial_excitation                                      # [...] Current-[F]
    departures = fields - cooling_fields[None, :]                             # [...] Scalar+[V]
    kinetic_energy = penetration_depth ** 2 * (ring_areas * (inertia(currents) & currents)).sites.sum()   # [...] Scalar+
    total_cost = (dual_areas * departures * departures).sites.sum() + kinetic_energy   # [...] Scalar+
    return candidate_quanta[total_cost.argmin(axis=0)]


retained_quanta = equilibrium_quanta(
    ring_topology, ring_areas, ring_gradients, in_hole,
    penetration_depth, candidate_quanta, cooling_fields,
)                                                                             # [n_cooling_fields]
ring_corners = ring_topology.incidence[2, 0]                                  # [3] Scalar+[F <- V]
ring_dual_areas = (ring_corners.adjugate() * (ring_areas / 3)).sum(axis=0)    # Scalar+[V]
zero_applied = np.zeros_like(retained_quanta, dtype=float)                    # [n_cooling_fields]
retained_fields = ring_fields(
    ring_topology, ring_areas, ring_gradients, in_hole,
    penetration_depth, retained_quanta, zero_applied,
)                                                                             # [n_cooling_fields] Scalar+[V]
retained_flux = (retained_fields * ring_dual_areas).sites.sum()               # [n_cooling_fields] Scalar+
render.staircase_figure(cooling_fields, retained_flux);

Retained magnetic flux rises in discrete unit steps as the cooling field increases, jumping to the next integer quantum whenever the applied flux across the hole and inner penetration skin crosses a half-integer threshold.

In [ ]:
# checks
corners = topology.incidence[2, 0]                                            # [3] Scalar+[F <- V]
assert (((gradients * (corners * heights)).sum(axis=0) - mv.y).norm() < 1e-10).all()

shallow_mask = (depths > 0.05) & (depths < 0.4)
depth_offsets = depths[shallow_mask] - depths[shallow_mask].mean()
log_offsets = strengths[shallow_mask].log() - strengths[shallow_mask].log().mean()
decay_length = -(depth_offsets * depth_offsets).sum() / (depth_offsets * log_offsets).sum()
assert ((decay_length - penetration_depth).abs() < 0.1 * penetration_depth).all()

core_mask = vertices.antinorm().batch() < 0.6
final_potentials = recorded_states[-1][0].batch()                             # [2, V] Scalar+
assert (final_potentials[0, core_mask].abs() < 0.02).all()
assert ((final_potentials[1, core_mask] - heights.batch()[core_mask]).abs() < 0.02).all()

trapped_flux = (trapped_fields * ring_dual_areas).sites.sum()                 # [3] Scalar+
assert ((trapped_flux - trapped_quanta).abs() < 0.02 * trapped_quanta).all()
assert set(np.unique(retained_quanta)) == {0, 1, 2}